# Rectangle packing with native QAOA

All 19 source stages, the 4 by 2 container and three rectangles, 17 active variables, 102 regenerated Ising terms, ten QAOA layers, 60 objective queries and 10,000 actual shots per query and final readout.

Native local COBYLA and the nonconstant-coefficient norm schedule reproduce the optimization setup at the original sizes. Finite sampling uses fixed local seeds. The final reported packing is a word observed in the final sample, and its cost is checked against the separate exact geometry table. Query traces describe the native run.

## Compile an explicit placement Hamiltonian

The visible program builder captures the regenerated 102-term placement
Observable, prepares seventeen decisions and composes ten cost/mixer layers.
FlagQuantum lowers this declared Pauli evolution and named target. The host
model chooses legal placements for three rectangles in a 4-by-2 container and
constructs overlap penalties; these geometric decisions are not inferred by
synth from arbitrary shapes.

Try editing a twenty-angle preview binding and compare the full cost with
the independently decoded packing. To alter the optimization algorithm too,
edit the shared quantum.py builder. Sixty objective queries with 10000 draws
each estimate mean cost; neither a low sampled mean nor one feasible placement
proves an optimal packing. The exact bounded cost remains a separate reference.


## Load the native packing workflow

In [ ]:
import math
from flagquantum import synth
from flagquantum.observables import I, Z
from flagquantum.synth import (
    RX,
    Output,
    QArray,
    QBit,
    allocate,
    hadamard_transform,
    qfunc,
    suzuki_trotter,
)
from examples.synthesis.applications.optimization.rectangles_packing.rectangles_packing_grid.overview.quantum import (
    Builder,
    PackingModel,
    NUM_QUBITS,
    LAYERS,
)


def program_builder(model: PackingModel) -> Builder:
    """Capture a static Observable while keeping all twenty angles bindable."""
    terms = model.terms
    operator = terms[0][1] * I()
    for support, coefficient in terms[1:]:
        word = I()
        for bit in support:
            word = word @ Z(bit)
        operator = operator + coefficient * word

    @qfunc
    def ansatz(angles: list[float], target: Output[QArray[QBit]]) -> None:
        allocate(17, target)
        hadamard_transform(target)
        for layer in range(10):
            suzuki_trotter(operator, angles[2 * layer], 1, 1, target)
            for bit in range(17):
                RX(angles[2 * layer + 1], target[bit])

    def build(angles: tuple[float, ...]) -> synth.Program:
        if len(angles) != 2 * LAYERS or not all(
            math.isfinite(value) for value in angles
        ):
            raise ValueError("The packing ansatz requires twenty finite angles")
        return synth.synthesize(
            ansatz,
            classical_inputs={"angles": angles},
            max_qubits=NUM_QUBITS,
            max_workspace=0,
            memory_limit_bytes=64 * 1024**2,
        )

    return build


from pathlib import Path

from IPython.display import Image, display

from examples.synthesis.applications.optimization.rectangles_packing.rectangles_packing_grid.overview.model import (
    HEIGHT,
    PENALTY,
    RECTANGLES,
    WIDTH,
    constraint_cost,
    make_model,
)
from examples.synthesis.applications.optimization.rectangles_packing.rectangles_packing_grid.overview.plots import (
    save_costs,
    save_packing,
    save_trace,
)
from examples.synthesis.applications.optimization.rectangles_packing.rectangles_packing_grid.overview.quantum import (
    LAYERS,
)
from examples.synthesis.applications.optimization.rectangles_packing.rectangles_packing_grid.overview.workflow import (
    MAX_QUERIES,
    SHOTS,
    initial_angles,
    run_workflow,
)

folder = Path("artifacts/development/rectangle-packing")


## Keep the original container and rectangle sizes

In [ ]:
print("Container and rectangles:", WIDTH, HEIGHT, RECTANGLES)


## Define the original placement constraints

In [ ]:
print("Each rectangle appears at most once, with disjoint cells and valid bounds.")


## Regenerate the fresh model

In [ ]:
model = make_model()


## Inspect bounded variables and terms

In [ ]:
print("Placement variables and Ising terms:", len(model.placements), len(model.terms))


## Keep all ten layers and the original penalty

In [ ]:
print("Layers and penalty:", LAYERS, PENALTY)


## Apply the finite optimizer cap

In [ ]:
print("Maximum objective queries:", MAX_QUERIES)


## Construct the original-size ansatz

In [ ]:
build = program_builder(model)


## Retain the actual sampling budget

In [ ]:
print("Shots per histogram:", SHOTS)


## Synthesize the initial norm-ramped circuit

In [ ]:
initial_program = build(initial_angles(model))
print("Initial resources:", initial_program.report["resources"])


## Run finite-sample mean optimization

In [ ]:
result = run_workflow()
print(
    "Actual objective queries and shots:",
    len(result.evaluations),
    len(result.evaluations) * SHOTS,
)
print("Final draws:", sum(count for _, count in result.sampled))
print("Best sampled mean:", result.selected.mean_cost)
print("Budget stop and convergence:", result.budget_reached, result.converged)


## Plot the observed optimizer trace

In [ ]:
display(Image(filename=str(save_trace(result, folder / "trace.png"))))


## Use the actual final sample

In [ ]:
assert len(result.sampled) > 0


## Plot the cost histogram

In [ ]:
display(Image(filename=str(save_costs(result, folder / "costs.png"))))


## Choose the best observed placement word

In [ ]:
print("Observed word and cost:", result.word, float(result.model.cost[result.word]))


## Preserve the fixed out-of-bounds variables as zero

In [ ]:
print("Active variables:", result.model.placements)


## Read the selected nonzero placements

In [ ]:
print("Observed placements:", result.placements)


## Draw the observed rectangle layout

In [ ]:
display(Image(filename=str(save_packing(result, folder / "packing.png"))))


## Check the geometry and exact reference

In [ ]:
print("Independent minimum cost:", result.exact_cost)
assert constraint_cost(result.placements) == float(result.model.cost[result.word])
print("Native resources:", result.program.report["resources"])
